# SaaSquatch DealRadar™ — Data Science & M&A Scoring Walkthrough
**Caprae Capital Partners | Full-Stack Developer Challenge**

This notebook demonstrates the core backend algorithms powering **SaaSquatch DealRadar™**:
1. **Multi-Source Entity Resolution & Deduplication** across Apollo, Google Maps, LinkedIn, Growjo, and Crunchbase (saving SaaSquatch enrichment credits).
2. **Defensible NAICS Revenue & EBITDA Triangulation** (detecting and correcting scraper revenue hallucinations).
3. **The Succession Risk Index™ (0–100)** to identify retiring *Silver Tsunami* founders with high transition intent.
4. **Post-Acquisition 7-Year AI Value Creation Scoring** aligned with Caprae Capital's operational transformation thesis.

In [ ]:
import csv
import re
from pathlib import Path

NAICS_BENCHMARKS = {
    '541512': {'title': 'Managed IT & MSP', 'rev_per_emp': 185000, 'ebitda_margin': 0.22},
    '238220': {'title': 'Commercial HVAC & Mechanical', 'rev_per_emp': 165000, 'ebitda_margin': 0.18},
    '811212': {'title': 'Data Center & Critical Infrastructure', 'rev_per_emp': 215000, 'ebitda_margin': 0.24},
    '488510': {'title': 'Specialized B2B Logistics', 'rev_per_emp': 210000, 'ebitda_margin': 0.16},
    '541380': {'title': 'Industrial Testing & Compliance', 'rev_per_emp': 175000, 'ebitda_margin': 0.23},
    '561720': {'title': 'Commercial Facility Maintenance', 'rev_per_emp': 115000, 'ebitda_margin': 0.17},
    '332710': {'title': 'Precision Manufacturing', 'rev_per_emp': 160000, 'ebitda_margin': 0.20},
}

def canonicalize_domain(url: str) -> str:
    url = (url or '').lower().strip()
    url = re.sub(r'^https?://', '', url)
    url = re.sub(r'^www\.', '', url)
    return url.split('/')[0]

csv_path = Path('../dataset/lmm_acquisition_leads.csv')
with open(csv_path, mode='r', encoding='utf-8') as f:
    raw_rows = list(csv.DictReader(f))

print(f'Loaded {len(raw_rows)} raw multi-source scraped records.')

In [ ]:
# Step 1: Multi-Source Entity Resolution & Deduplication
golden_records = {}
duplicates_suppressed = 0

for row in raw_rows:
    domain = canonicalize_domain(row['website'])
    if domain not in golden_records:
        row['canonical_domain'] = domain
        row['sources'] = [row['source']]
        row['merged_count'] = 0
        golden_records[domain] = row
    else:
        duplicates_suppressed += 1
        existing = golden_records[domain]
        existing['merged_count'] += 1
        if row['source'] not in existing['sources']:
            existing['sources'].append(row['source'])
        # Prefer direct executive email over generic info@
        if existing['owner_email'].startswith('info@') and not row['owner_email'].startswith('info@'):
            existing['owner_email'] = row['owner_email']

print(f'Golden Records: {len(golden_records)} | Duplicates Merged: {duplicates_suppressed} (+{duplicates_suppressed} SaaSquatch Credits Saved)')

In [ ]:
# Step 2: Succession Risk Index & NAICS Revenue Triangulation
scored_targets = []
for domain, rec in golden_records.items():
    emp = int(rec['employee_count'])
    tenure = int(rec['owner_tenure_years'])
    age_yrs = 2026 - int(rec['founded_year'])
    naics = NAICS_BENCHMARKS[rec['naics_code']]
    
    naics_rev = emp * naics['rev_per_emp']
    raw_rev = float(rec['reported_revenue_usd']) if rec['reported_revenue_usd'] else naics_rev
    # Guard against >55% scraper hallucination spikes
    if abs(raw_rev / naics_rev - 1.0) > 0.55:
        triangulated_rev = 0.75 * naics_rev + 0.25 * raw_rev
    else:
        triangulated_rev = 0.55 * naics_rev + 0.45 * raw_rev
        
    est_ebitda = triangulated_rev * naics['ebitda_margin']
    
    # Succession Risk Score (0-100)
    score = 0
    score += 35 if tenure >= 25 else (27 if tenure >= 18 else 5)
    score += 25 if rec['has_second_tier_mgmt'] == 'false' else 12
    score += 20 if (2026 - int(rec['website_last_updated'])) >= 5 else 6
    score += 20 if rec['ownership_type'] == 'Founder / Family Owned' else 0
    
    scored_targets.append({
        'company': rec['company_name'],
        'sources': '+'.join(rec['sources']),
        'succession_score': score,
        'triangulated_rev_m': round(triangulated_rev / 1e6, 2),
        'est_ebitda_m': round(est_ebitda / 1e6, 2),
        'ownership': rec['ownership_type']
    })

scored_targets.sort(key=lambda x: x['succession_score'], reverse=True)
for t in scored_targets[:8]:
    print(f"{t['company']} [{t['sources']}] -> Succession: {t['succession_score']}/100 | Rev: ${t['triangulated_rev_m']}M | EBITDA: ${t['est_ebitda_m']}M")